# dashboard static page v1

目标：验证 `src/dashboard` 能把 `dashboard.view_model.v1` 渲染成可直接打开的 HTML 页面，展示模型回测、regime/window-regime 差异、fidelity 和 attribution。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'SPEC.md').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('cannot locate repo root from notebook cwd')
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dashboard import load_dashboard_view_model, write_dashboard_html

CARD = Path('research/dashboard_static_page_v1')
SOURCE_EXPORT = Path('research/schedule_dashboard_attribution_refresh_v1/dashboard_export.json')
OUTPUT_HTML = CARD / 'dashboard.html'
CARD.mkdir(parents=True, exist_ok=True)

{'source_export': str(SOURCE_EXPORT), 'output_html': str(OUTPUT_HTML)}

{'source_export': 'research/schedule_dashboard_attribution_refresh_v1/dashboard_export.json',
 'output_html': 'research/dashboard_static_page_v1/dashboard.html'}

## 生成页面

In [2]:
view_model = load_dashboard_view_model(SOURCE_EXPORT)
html = write_dashboard_html(view_model, output_path=OUTPUT_HTML, title='SOXL 模型回测 Dashboard')
html[:500]

'<!doctype html>\n<html lang="zh-CN">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1">\n  <title>SOXL 模型回测 Dashboard</title>\n  <style>\n    :root { color-scheme: light; --bg:#f7f8fa; --panel:#ffffff; --ink:#1f2933; --muted:#667085; --line:#d8dee8; --accent:#1b6b93; --warn:#8a4b0f; --good:#126b45; }\n    * { box-sizing: border-box; }\n    body { margin: 0; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; background: var(--bg);'

## 验证页面结构

In [3]:
summary = {
    'schema_version': 'dashboard.static_page_evidence.v1',
    'status': 'completed',
    'source_export': str(SOURCE_EXPORT),
    'output_html': str(OUTPUT_HTML),
    'html_size_bytes': OUTPUT_HTML.stat().st_size,
    'model_card_count': html.count('class="model-card'),
    'has_model_filter': 'id="model-filter"' in html,
    'has_regime_filter': 'id="regime-filter"' in html,
    'has_fidelity_filter': 'id="fidelity-filter"' in html,
    'has_attribution_filter': 'id="attribution-filter"' in html,
    'has_embedded_view_model': 'id="dashboard-view-model"' in html,
    'has_window_regime_table': 'window-regime' in html,
    'visible_models': [card['model'] for card in view_model['model_cards']],
    'controls': view_model['controls'],
    'risk_notes': [
        '静态页面消费 view model，不直接读取实验 sidecar。',
        '页面内置简单筛选脚本，后续可替换为更完整前端应用。',
    ],
}
assert summary['model_card_count'] == len(view_model['model_cards'])
assert summary['has_model_filter']
assert summary['has_regime_filter']
assert summary['has_fidelity_filter']
assert summary['has_attribution_filter']
assert summary['has_embedded_view_model']
assert summary['has_window_regime_table']
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'dashboard.static_page_evidence.v1',
 'status': 'completed',
 'source_export': 'research/schedule_dashboard_attribution_refresh_v1/dashboard_export.json',
 'output_html': 'research/dashboard_static_page_v1/dashboard.html',
 'html_size_bytes': 126816,
 'model_card_count': 5,
 'has_model_filter': True,
 'has_regime_filter': True,
 'has_fidelity_filter': True,
 'has_attribution_filter': True,
 'has_embedded_view_model': True,
 'has_window_regime_table': True,
 'visible_models': ['ridge_mse__top_ic_factors',
  'gcn_mse__top_ic_factors',
  'gcn_mse__ic_weighted_composite',
  'ridge_mse__ic_weighted_composite',
  'ridge_mse__all_features'],
 'controls': {'models': ['gcn_mse__ic_weighted_composite',
   'gcn_mse__top_ic_factors',
   'ridge_mse__all_features',
   'ridge_mse__ic_weighted_composite',
   'ridge_mse__top_ic_factors'],
  'base_models': ['gcn_mse', 'ridge_mse'],
  'feature_sets': ['all_features', 'ic_weighted_composite', 'top_ic_factors'],
  'regimes': ['bull', 'do

## 中文结论

- 做了什么：使用 `src/dashboard` 的 view model 和 HTML renderer 生成 `dashboard.html`。
- 核心指标：页面包含模型、regime、fidelity、attribution 筛选控件，模型卡片数量与 view model 一致。
- regime 差异：页面模型卡片包含 Regime 和 Window-Regime 表格，可展示不同市场状态下的回测差异。
- 是否晋升：不晋升 production；这是 dashboard 页面能力验证。
- 风险：当前是静态 HTML，交互脚本只做客户端筛选；生产页面后续应接入同一个 `dashboard.view_model.v1`，避免页面读取散落实验文件。
- 下一步：把 schedule refresh 末端增加可选 HTML page export，形成 matrix -> dashboard export -> view model -> page 的自动刷新链路。